In [ ]:
import pennylane as qml
import matplotlib.pyplot as plt
import numpy as np
import torch

In [ ]:
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def quantum_projection_circuit(inputs, weights):
    inputs=torch.from_numpy(inputs)
    for i in range(n_qubits):
        qml.Hadamard(wires=i)
        feature = inputs[..., i]
        qml.RY(torch.arctan(feature), wires=i)
        qml.RZ(torch.arctan(feature**2), wires=i)
    
    n_layers = weights.shape[0] if len(weights.shape) > 2 else 1
    
    for l in range(n_layers):
        cnot_pairs = [
            (0, 1), (1, 2), (2, 3), (3, 0),
            (0, 2), (1, 3), (2, 0), (3, 1)
        ]

        for ctrl, tgt in cnot_pairs:
            qml.CNOT(wires=[ctrl, tgt])
            
        for i in range(n_qubits):
            qml.Rot(weights[l, i, 0], weights[l, i, 1], weights[l, i, 2], wires=i)
    
    # 3. Measurement
    return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]

In [ ]:
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

DEPOLARIZING_PROB = 0.01
AMPLITUDE_DAMPING_GAMMA = 0.005

@qml.qnode(dev)
def quantum_projection_circuit(inputs, weights):
    inputs=torch.from_numpy(inputs)
    for i in range(n_qubits):
        qml.Hadamard(wires=i)
        feature = inputs[..., i]
        qml.RY(torch.arctan(feature), wires=i)
        qml.RZ(torch.arctan(feature**2), wires=i)
        qml.DepolarizingChannel(DEPOLARIZING_PROB, wires=i)
        qml.AmplitudeDamping(AMPLITUDE_DAMPING_GAMMA, wires=i)
    
    n_layers = weights.shape[0] if len(weights.shape) > 2 else 1
    
    for l in range(n_layers):

        cnot_pairs = [
            (0, 1), (1, 2), (2, 3), (3, 0),
            (0, 2), (1, 3), (2, 0), (3, 1)
        ]

        for ctrl, tgt in cnot_pairs:
            qml.CNOT(wires=[ctrl, tgt])
            qml.DepolarizingChannel(DEPOLARIZING_PROB * 2, wires=ctrl)
            qml.DepolarizingChannel(DEPOLARIZING_PROB * 2, wires=tgt)
            
        for i in range(n_qubits):
            qml.Rot(weights[l, i, 0], weights[l, i, 1], weights[l, i, 2], wires=i)
            qml.DepolarizingChannel(DEPOLARIZING_PROB, wires=i)
            qml.AmplitudeDamping(AMPLITUDE_DAMPING_GAMMA, wires=i)
    
    return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]

In [ ]:
n_layers=1
fig, ax = qml.draw_mpl(quantum_projection_circuit)(np.random.random(n_qubits),np.random.random((n_layers,n_qubits,3)))
plt.show()